In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/health.db")
print("Connected!")

Connected!


In [2]:
con.execute("SELECT * FROM patients LIMIT 5").df()

,patient_id,first_name,last_name,gender,birth_date,age,city,state,postal_code,insurance
0,4989a96a-dc04-49e4-9b9b-baf21ed26eb4,Angel,Hill,Male,1950-08-03,75,East Jill,Utah,97581,Medicare
1,b8d0b0ba-0766-4846-b42e-063ef0af83b8,William,Johnson,Male,1967-02-06,59,Lake Debra,Ohio,55488,Private
2,1939f749-398f-4d1d-b969-c53cc8649cfa,Katherine,Moore,Female,1936-04-27,90,Petersonberg,Indiana,44619,Medicaid
3,a39fb550-3f9d-4038-b989-402f30f9cf6d,Cassandra,Roman,Male,1982-04-01,44,Herrerafurt,Colorado,72858,Private
4,73030d01-3138-4009-a944-045cb826dd55,Donna,Mejia,Female,1941-09-02,84,Port Jesseville,Missouri,36935,Uninsured


In [3]:
con.execute("""
    SELECT
        CASE
            WHEN age < 30 THEN 'Under 30'
            WHEN age < 50 THEN '30 to 49'
            WHEN age < 70 THEN '50 to 69'
            ELSE '70 and over'
        END AS age_group,
        COUNT(*) AS patient_count
    FROM patients
    GROUP BY age_group
    ORDER BY age_group
""").df()

,age_group,patient_count
0,30 to 49,282
1,50 to 69,281
2,70 and over,274
3,Under 30,163


In [4]:
con.execute("""
    SELECT diagnosis, COUNT(*) AS total
    FROM conditions
    GROUP BY diagnosis
    ORDER BY total DESC
""").df()

,diagnosis,total
0,Hypertension,195
1,Coronary Artery Disease,193
2,Type 2 Diabetes Mellitus,179
3,Sepsis,178
4,COPD,177
5,Stroke,174
6,Heart Failure,174
7,Pneumonia,169
8,Obesity,168
9,Depression,168


In [5]:
con.execute("""
    SELECT
        department,
        ROUND(AVG(length_of_stay), 1) AS avg_days,
        COUNT(*) AS total_visits
    FROM encounters
    GROUP BY department
    ORDER BY avg_days DESC
""").df()

,department,avg_days,total_visits
0,Cardiology,7.8,630
1,General Medicine,7.8,611
2,Pulmonology,7.5,591
3,ICU,7.3,569
4,Nephrology,7.2,570


In [6]:
con.execute("""
    SELECT
        p.first_name || ' ' || p.last_name AS patient_name,
        p.age,
        p.insurance,
        COUNT(e.encounter_id) AS total_visits
    FROM patients p
    JOIN encounters e ON p.patient_id = e.patient_id
    GROUP BY p.patient_id, patient_name, p.age, p.insurance
    ORDER BY total_visits DESC
    LIMIT 10
""").df()

,patient_name,age,insurance,total_visits
0,Sarah Johnson,67,Private,5
1,Richard Graham,38,Private,5
2,Joshua Smith,69,Medicaid,5
3,Debra Wang,55,Medicaid,5
4,Matthew Adams,53,Medicaid,5
5,Michelle Coleman,62,Medicaid,5
6,Brian Watson,53,Medicaid,5
7,Beth Williams,57,Private,5
8,Jessica Costa,31,Medicare,5
9,Pamela Harper,30,Medicare,5
